In [1]:
practice_attempts_df = spark.table(
    "demo.silver.practice_attempts"
)

question_bank_df = spark.table(
    "demo.silver.question_bank"
)

taxonomy_df = spark.table(
    "demo.silver.content_taxonomy"
)

print("Practice attempts:", practice_attempts_df.count())
print("Questions:", question_bank_df.count())
print("Taxonomy records:", taxonomy_df.count())

Practice attempts: 5
Questions: 5
Taxonomy records: 10


In [2]:
practice_attempts_df.select(
    "attempt_id",
    "event_id",
    "user_id",
    "session_id",
    "question_id",
    "question_version",
    "attempt_time",
    "is_correct",
    "score",
    "hints_used",
    "attempt_duration_seconds",
    "attempt_number"
).show(truncate=False)

question_bank_df.select(
    "question_id",
    "question_version",
    "domain",
    "topic",
    "subtopic"
).show(truncate=False)

+----------------------------------------------------------------+--------+--------+-----------+------------+----------------+-------------------+----------+-----+----------+------------------------+--------------+
|attempt_id                                                      |event_id|user_id |session_id |question_id |question_version|attempt_time       |is_correct|score|hints_used|attempt_duration_seconds|attempt_number|
+----------------------------------------------------------------+--------+--------+-----------+------------+----------------+-------------------+----------+-----+----------+------------------------+--------------+
|fb827af1d27682d1cc5d74c5cd4ea504e5514853143ad2d50d9a6406ad89c514|evt_0002|user_001|session_001|question_001|1               |2026-07-20 09:12:00|false     |0.0  |1         |80                      |1             |
|d76fe2bf4801f31255fe5f66181612e6d4d8fa3c5c7ffe736c2ca01bb77c485c|evt_0002|user_001|session_001|question_002|1               |2026-07-20 09:

In [3]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    regexp_replace
)

def normalize_column(column):
    return lower(
        trim(
            regexp_replace(column, r"\s+", " ")
        )
    )

practice_with_question_df = (
    practice_attempts_df.alias("p")
    .join(
        question_bank_df.alias("q"),
        (
            col("p.question_id") == col("q.question_id")
        )
        & (
            col("p.question_version") == col("q.question_version")
        ),
        "left"
    )
    .select(
        col("p.attempt_id"),
        col("p.event_id"),
        col("p.user_id"),
        col("p.session_id"),
        col("p.question_id"),
        col("p.question_version"),
        col("p.attempt_time"),
        col("p.is_correct"),
        col("p.score"),
        col("p.hints_used"),
        col("p.attempt_duration_seconds"),
        col("p.attempt_number"),

        col("q.domain"),
        col("q.topic"),
        col("q.subtopic"),

        normalize_column(col("q.domain")).alias(
            "normalized_domain"
        ),
        normalize_column(col("q.topic")).alias(
            "normalized_topic"
        ),
        normalize_column(col("q.subtopic")).alias(
            "normalized_subtopic"
        )
    )
)

In [4]:
practice_taxonomy_matches_df = (
    practice_with_question_df.alias("p")
    .join(
        taxonomy_df
        .filter(col("taxonomy_level") == "subtopic")
        .alias("t"),
        (
            col("p.normalized_domain")
            == col("t.normalized_domain")
        )
        & (
            col("p.normalized_topic")
            == col("t.normalized_topic")
        )
        & (
            col("p.normalized_subtopic")
            == col("t.normalized_subtopic")
        ),
        "left"
    )
    .select(
        col("p.attempt_id"),
        col("p.event_id"),
        col("p.user_id"),
        col("p.session_id"),
        col("p.question_id"),
        col("p.question_version"),
        col("p.attempt_time"),
        col("p.is_correct"),
        col("p.score"),
        col("p.hints_used"),
        col("p.attempt_duration_seconds"),
        col("p.attempt_number"),

        col("p.domain"),
        col("p.topic"),
        col("p.subtopic"),

        col("t.taxonomy_id"),
        col("t.taxonomy_level"),
        col("t.validation_status").alias(
            "taxonomy_validation_status"
        )
    )
)

In [5]:
practice_taxonomy_matches_df.select(
    "attempt_id",
    "user_id",
    "question_id",
    "domain",
    "topic",
    "subtopic",
    "taxonomy_id",
    "taxonomy_level",
    "taxonomy_validation_status"
).show(truncate=False)

+----------------------------------------------------------------+--------+------------+----------------+-----------------+---------------------+----------------------------------------------------------------+--------------+--------------------------+
|attempt_id                                                      |user_id |question_id |domain          |topic            |subtopic             |taxonomy_id                                                     |taxonomy_level|taxonomy_validation_status|
+----------------------------------------------------------------+--------+------------+----------------+-----------------+---------------------+----------------------------------------------------------------+--------------+--------------------------+
|fb827af1d27682d1cc5d74c5cd4ea504e5514853143ad2d50d9a6406ad89c514|user_001|question_001|Computer Science|Operating Systems|Virtual Memory       |6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|subtopic      |approved        

In [6]:
practice_taxonomy_matches_df.select(
    col("attempt_id")
).filter(
    col("taxonomy_id").isNull()
).show(truncate=False)

+----------+
|attempt_id|
+----------+
+----------+



In [7]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

practice_evidence_df = (
    practice_taxonomy_matches_df

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("practice_attempt"),
                col("attempt_id"),
                col("taxonomy_id")
            ),
            256
        )
    )

    .withColumn(
        "evidence_type",
        lit("practice_attempt")
    )

    .withColumn(
        "evidence_time",
        col("attempt_time")
    )

    .withColumn(
        "feedback_id",
        lit(None).cast("string")
    )

    .withColumn(
        "insight_id",
        lit(None).cast("string")
    )

    .withColumn(
        "validation_id",
        lit(None).cast("string")
    )

    .withColumn(
        "extraction_confidence",
        lit(None).cast("float")
    )

    .withColumn(
        "semantic_match_score",
        lit(None).cast("float")
    )

    .withColumn(
        "reliability_score",
        lit(None).cast("float")
    )

    .withColumn(
        "contradiction_flag",
        lit(None).cast("boolean")
    )

    .withColumn(
        "confidence_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_understanding_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_difficulty_score",
        lit(None).cast("int")
    )

    .withColumn(
        "still_confused",
        lit(None).cast("boolean")
    )

    .withColumn(
        "source_table",
        lit("silver_practice_attempts")
    )

    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        "evidence_id",
        "user_id",
        "session_id",
        "taxonomy_id",

        "event_id",
        "attempt_id",
        "feedback_id",
        "insight_id",
        "validation_id",

        "evidence_type",
        "evidence_time",

        "is_correct",
        "score",
        "hints_used",
        "attempt_duration_seconds",
        "attempt_number",

        "extraction_confidence",
        "semantic_match_score",
        "reliability_score",
        "contradiction_flag",

        "confidence_score",
        "perceived_understanding_score",
        "perceived_difficulty_score",
        "still_confused",

        "source_table",
        "processing_time"
    )
)

In [8]:
practice_evidence_df.select(
    "evidence_id",
    "user_id",
    "session_id",
    "taxonomy_id",
    "event_id",
    "attempt_id",
    "evidence_type",
    "evidence_time",
    "is_correct",
    "score",
    "hints_used",
    "attempt_duration_seconds",
    "attempt_number",
    "source_table"
).show(truncate=False)

print(
    "Practice evidence rows:",
    practice_evidence_df.count()
)

print(
    "Distinct evidence IDs:",
    practice_evidence_df
    .select("evidence_id")
    .distinct()
    .count()
)

+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+----------------+-------------------+----------+-----+----------+------------------------+--------------+------------------------+
|evidence_id                                                     |user_id |session_id |taxonomy_id                                                     |event_id|attempt_id                                                      |evidence_type   |evidence_time      |is_correct|score|hints_used|attempt_duration_seconds|attempt_number|source_table            |
+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+----------------+-------------------+----------+-----+------

In [9]:
ai_insights_df = spark.table(
    "demo.silver.ai_extracted_insights"
)

ai_insights_df.select(
    "insight_id",
    "event_id",
    "user_id",
    "session_id",
    "dynamic_concept_name",
    "extracted_at",
    "extraction_confidence",
    "validation_status"
).show(truncate=False)

+----------------------------------------------------------------+--------+--------+-----------+--------------------+-------------------+---------------------+-----------------+
|insight_id                                                      |event_id|user_id |session_id |dynamic_concept_name|extracted_at       |extraction_confidence|validation_status|
+----------------------------------------------------------------+--------+--------+-----------+--------------------+-------------------+---------------------+-----------------+
|a812e708876c32b4bb8d415842bf2de7a9114b7b88efe690f0c36a66875f65df|evt_0001|user_001|session_001|Operating Systems   |2026-07-20 09:00:08|1.0                  |pending          |
|66317e7e2b57674ab50b4418ee576ccc39e490ca3f3f5009cd3ad2c3d474a3fe|evt_0001|user_001|session_001|Virtual Memory      |2026-07-20 09:00:08|1.0                  |pending          |
|a6cb2c3a4bd19f4a389472160bd13969533dbe49e042f7855ab5ac61c66bfab6|evt_0001|user_001|session_001|Page Fault    

In [14]:
from pyspark.sql.functions import when

In [15]:
taxonomy_matchable_df = (
    taxonomy_df
    .filter(
        col("taxonomy_level").isin(
            "topic",
            "subtopic",
            "concept"
        )
    )
    .withColumn(
        "normalized_match_name",
        when(
            col("taxonomy_level") == "topic",
            col("normalized_topic")
        )
        .when(
            col("taxonomy_level") == "subtopic",
            col("normalized_subtopic")
        )
        .otherwise(
            col("normalized_concept_name")
        )
    )
    .select(
        "taxonomy_id",
        "taxonomy_level",
        "validation_status",
        "normalized_match_name"
    )
)

In [16]:
ai_taxonomy_matches_df = (
    ai_insights_df.alias("ai")
    .withColumn(
        "normalized_dynamic_concept",
        normalize_column(
            col("dynamic_concept_name")
        )
    )
    .join(
        taxonomy_matchable_df.alias("t"),
        col("normalized_dynamic_concept")
        == col("t.normalized_match_name"),
        "left"
    )
    .select(
        col("ai.insight_id"),
        col("ai.event_id"),
        col("ai.user_id"),
        col("ai.session_id"),
        col("ai.dynamic_concept_name"),
        col("ai.extracted_at"),
        col("ai.extraction_confidence"),
        col("ai.validation_status").alias(
            "insight_validation_status"
        ),
        col("t.taxonomy_id"),
        col("t.taxonomy_level"),
        col("t.validation_status").alias(
            "taxonomy_validation_status"
        )
    )
)

In [17]:
ai_taxonomy_matches_df.select(
    "insight_id",
    "dynamic_concept_name",
    "taxonomy_id",
    "taxonomy_level",
    "taxonomy_validation_status"
).orderBy(
    "dynamic_concept_name"
).show(truncate=False)

+----------------------------------------------------------------+--------------------+----------------------------------------------------------------+--------------+--------------------------+
|insight_id                                                      |dynamic_concept_name|taxonomy_id                                                     |taxonomy_level|taxonomy_validation_status|
+----------------------------------------------------------------+--------------------+----------------------------------------------------------------+--------------+--------------------------+
|1e03e8cbacf3d9cbfae14755113f800a3e50e69859b3dd8bcb902458d265dfa9|Base Case           |3b4c81d554cb203b67ef6add2b90412abb55e91dbc234073e055d8e5ee7b34fe|concept       |approved                  |
|7215f4912ad030943daea0c55e0341db76874b42ea87bbb5011537e664c8d3e2|Memory Layout       |50989f4e7b13c6b4bae1b55e62cce73aadc67b8fd4fdf694fd562445adc8bb87|concept       |pending                   |
|a812e708876c32b4bb8d4158

In [18]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

ai_insight_evidence_df = (
    ai_taxonomy_matches_df

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("ai_insight"),
                col("insight_id"),
                col("taxonomy_id")
            ),
            256
        )
    )

    .withColumn(
        "attempt_id",
        lit(None).cast("string")
    )

    .withColumn(
        "feedback_id",
        lit(None).cast("string")
    )

    .withColumn(
        "validation_id",
        lit(None).cast("string")
    )

    .withColumn(
        "evidence_type",
        lit("ai_insight")
    )

    .withColumn(
        "evidence_time",
        col("extracted_at")
    )

    .withColumn(
        "is_correct",
        lit(None).cast("boolean")
    )

    .withColumn(
        "score",
        lit(None).cast("float")
    )

    .withColumn(
        "hints_used",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_duration_seconds",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_number",
        lit(None).cast("int")
    )

    .withColumn(
        "semantic_match_score",
        lit(None).cast("float")
    )

    .withColumn(
        "reliability_score",
        lit(None).cast("float")
    )

    .withColumn(
        "contradiction_flag",
        lit(None).cast("boolean")
    )

    .withColumn(
        "confidence_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_understanding_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_difficulty_score",
        lit(None).cast("int")
    )

    .withColumn(
        "still_confused",
        lit(None).cast("boolean")
    )

    .withColumn(
        "source_table",
        lit("silver_ai_extracted_insights")
    )

    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        "evidence_id",
        "user_id",
        "session_id",
        "taxonomy_id",

        "event_id",
        "attempt_id",
        "feedback_id",
        "insight_id",
        "validation_id",

        "evidence_type",
        "evidence_time",

        "is_correct",
        "score",
        "hints_used",
        "attempt_duration_seconds",
        "attempt_number",

        "extraction_confidence",
        "semantic_match_score",
        "reliability_score",
        "contradiction_flag",

        "confidence_score",
        "perceived_understanding_score",
        "perceived_difficulty_score",
        "still_confused",

        "source_table",
        "processing_time"
    )
)

In [19]:
ai_insight_evidence_df.select(
    "evidence_id",
    "user_id",
    "session_id",
    "taxonomy_id",
    "event_id",
    "insight_id",
    "evidence_type",
    "evidence_time",
    "extraction_confidence",
    "source_table"
).show(truncate=False)

print(
    "AI insight evidence rows:",
    ai_insight_evidence_df.count()
)

print(
    "Distinct evidence IDs:",
    ai_insight_evidence_df
    .select("evidence_id")
    .distinct()
    .count()
)

+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+-------------+-------------------+---------------------+----------------------------+
|evidence_id                                                     |user_id |session_id |taxonomy_id                                                     |event_id|insight_id                                                      |evidence_type|evidence_time      |extraction_confidence|source_table                |
+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+-------------+-------------------+---------------------+----------------------------+
|8abbbdc4f2d67efbe4bae5a83b7b7fe6116f00bc6011b9ca2cc37a5e036cb3d

In [20]:
validated_insights_df = spark.table(
    "demo.silver.validated_learning_insights"
)

validated_with_ai_df = (
    validated_insights_df.alias("v")
    .join(
        ai_insights_df.alias("ai"),
        col("v.insight_id") == col("ai.insight_id"),
        "inner"
    )
    .join(
        taxonomy_matchable_df.alias("t"),
        normalize_column(
            col("v.dynamic_concept_name")
        ) == col("t.normalized_match_name"),
        "left"
    )
    .select(
        col("v.validation_id"),
        col("v.insight_id"),
        col("ai.event_id"),
        col("v.user_id"),
        col("v.session_id"),
        col("v.dynamic_concept_name"),
        col("v.validation_time"),
        col("ai.extraction_confidence"),
        col("v.semantic_match_score"),
        col("v.reliability_score"),
        col("v.contradiction_flag"),
        col("v.validation_status"),
        col("t.taxonomy_id"),
        col("t.taxonomy_level")
    )
)

In [21]:
validated_with_ai_df.select(
    "validation_id",
    "insight_id",
    "dynamic_concept_name",
    "taxonomy_id",
    "taxonomy_level",
    "semantic_match_score",
    "reliability_score",
    "contradiction_flag",
    "validation_status"
).orderBy(
    "dynamic_concept_name"
).show(truncate=False)

+----------------------------------------------------------------+----------------------------------------------------------------+--------------------+----------------------------------------------------------------+--------------+--------------------+-----------------+------------------+-----------------+
|validation_id                                                   |insight_id                                                      |dynamic_concept_name|taxonomy_id                                                     |taxonomy_level|semantic_match_score|reliability_score|contradiction_flag|validation_status|
+----------------------------------------------------------------+----------------------------------------------------------------+--------------------+----------------------------------------------------------------+--------------+--------------------+-----------------+------------------+-----------------+
|d5dd866469b673a37c89caa6a128ba5708aaf922d4a1b970e74aa075cb8c8049|1e03e8c

In [22]:
validated_with_ai_df.filter(
    col("taxonomy_id").isNull()
).select(
    "validation_id",
    "dynamic_concept_name"
).show(truncate=False)

+-------------+--------------------+
|validation_id|dynamic_concept_name|
+-------------+--------------------+
+-------------+--------------------+



In [23]:
validated_insight_evidence_df = (
    validated_with_ai_df

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("validated_insight"),
                col("validation_id"),
                col("taxonomy_id")
            ),
            256
        )
    )

    .withColumn(
        "attempt_id",
        lit(None).cast("string")
    )

    .withColumn(
        "feedback_id",
        lit(None).cast("string")
    )

    .withColumn(
        "evidence_type",
        lit("validated_insight")
    )

    .withColumn(
        "evidence_time",
        col("validation_time")
    )

    .withColumn(
        "is_correct",
        lit(None).cast("boolean")
    )

    .withColumn(
        "score",
        lit(None).cast("float")
    )

    .withColumn(
        "hints_used",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_duration_seconds",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_number",
        lit(None).cast("int")
    )

    .withColumn(
        "confidence_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_understanding_score",
        lit(None).cast("int")
    )

    .withColumn(
        "perceived_difficulty_score",
        lit(None).cast("int")
    )

    .withColumn(
        "still_confused",
        lit(None).cast("boolean")
    )

    .withColumn(
        "source_table",
        lit("silver_validated_learning_insights")
    )

    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        "evidence_id",
        "user_id",
        "session_id",
        "taxonomy_id",

        "event_id",
        "attempt_id",
        "feedback_id",
        "insight_id",
        "validation_id",

        "evidence_type",
        "evidence_time",

        "is_correct",
        "score",
        "hints_used",
        "attempt_duration_seconds",
        "attempt_number",

        "extraction_confidence",
        "semantic_match_score",
        "reliability_score",
        "contradiction_flag",

        "confidence_score",
        "perceived_understanding_score",
        "perceived_difficulty_score",
        "still_confused",

        "source_table",
        "processing_time"
    )
)

In [24]:
validated_insight_evidence_df.select(
    "evidence_id",
    "user_id",
    "session_id",
    "taxonomy_id",
    "event_id",
    "insight_id",
    "validation_id",
    "evidence_type",
    "evidence_time",
    "extraction_confidence",
    "semantic_match_score",
    "reliability_score",
    "contradiction_flag",
    "source_table"
).show(truncate=False)

print(
    "Validated insight evidence rows:",
    validated_insight_evidence_df.count()
)

print(
    "Distinct evidence IDs:",
    validated_insight_evidence_df
    .select("evidence_id")
    .distinct()
    .count()
)

+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+----------------------------------------------------------------+-----------------+--------------------------+---------------------+--------------------+-----------------+------------------+----------------------------------+
|evidence_id                                                     |user_id |session_id |taxonomy_id                                                     |event_id|insight_id                                                      |validation_id                                                   |evidence_type    |evidence_time             |extraction_confidence|semantic_match_score|reliability_score|contradiction_flag|source_table                      |
+----------------------------------------------------------------+--------+-----------+---------

In [25]:
pre_feedback_df = spark.table(
    "demo.silver.pre_practice_feedback"
)

post_feedback_df = spark.table(
    "demo.silver.post_practice_feedback"
)

pre_feedback_df.select(
    "feedback_id",
    "user_id",
    "session_id",
    "practice_id",
    "feedback_time",
    "confidence_before_score",
    "perceived_understanding_before_score",
    "expected_difficulty_score"
).show(truncate=False)

post_feedback_df.select(
    "feedback_id",
    "user_id",
    "session_id",
    "practice_id",
    "feedback_time",
    "confidence_after_score",
    "perceived_understanding_after_score",
    "perceived_difficulty_score",
    "still_confused"
).show(truncate=False)

+------------+--------+-----------+------------+-------------------+-----------------------+------------------------------------+-------------------------+
|feedback_id |user_id |session_id |practice_id |feedback_time      |confidence_before_score|perceived_understanding_before_score|expected_difficulty_score|
+------------+--------+-----------+------------+-------------------+-----------------------+------------------------------------+-------------------------+
|feedback_001|user_001|session_001|practice_001|2026-07-20 09:04:00|4                      |3                                   |8                        |
|feedback_003|user_002|session_002|practice_002|2026-07-21 11:04:00|9                      |9                                   |3                        |
|feedback_005|user_003|session_003|practice_003|2026-07-22 14:04:00|7                      |7                                   |4                        |
+------------+--------+-----------+------------+----------------

In [26]:
practice_topics_df = (
    practice_attempts_df.alias("p")
    .join(
        question_bank_df.alias("q"),
        (
            col("p.question_id") == col("q.question_id")
        )
        & (
            col("p.question_version") == col("q.question_version")
        ),
        "inner"
    )
    .select(
        col("p.practice_id"),
        col("p.user_id"),
        col("p.session_id"),
        col("q.domain"),
        col("q.topic"),
        col("q.subtopic")
    )
    .distinct()
)

In [27]:
practice_topics_df.orderBy(
    "practice_id",
    "domain",
    "topic",
    "subtopic"
).show(truncate=False)

+------------+--------+-----------+----------------+-----------------+---------------------+
|practice_id |user_id |session_id |domain          |topic            |subtopic             |
+------------+--------+-----------+----------------+-----------------+---------------------+
|practice_001|user_001|session_001|Computer Science|Operating Systems|Virtual Memory       |
|practice_002|user_002|session_002|Computer Science|Programming      |Recursion            |
|practice_003|user_003|session_003|Computer Science|Operating Systems|Process Memory Layout|
+------------+--------+-----------+----------------+-----------------+---------------------+



In [28]:
from pyspark.sql.functions import countDistinct

practice_topics_df.groupBy(
    "practice_id"
).agg(
    countDistinct(
        "domain",
        "topic",
        "subtopic"
    ).alias("distinct_taxonomy_items")
).orderBy(
    "practice_id"
).show()

+------------+-----------------------+
| practice_id|distinct_taxonomy_items|
+------------+-----------------------+
|practice_001|                      1|
|practice_002|                      1|
|practice_003|                      1|
+------------+-----------------------+



In [29]:
practice_taxonomy_df = (
    practice_topics_df.alias("p")
    .join(
        taxonomy_df
        .filter(col("taxonomy_level") == "subtopic")
        .alias("t"),
        (
            normalize_column(col("p.domain"))
            == col("t.normalized_domain")
        )
        & (
            normalize_column(col("p.topic"))
            == col("t.normalized_topic")
        )
        & (
            normalize_column(col("p.subtopic"))
            == col("t.normalized_subtopic")
        ),
        "left"
    )
    .select(
        col("p.practice_id"),
        col("p.user_id"),
        col("p.session_id"),
        col("t.taxonomy_id")
    )
)

In [30]:
practice_taxonomy_df.show(truncate=False)

+------------+--------+-----------+----------------------------------------------------------------+
|practice_id |user_id |session_id |taxonomy_id                                                     |
+------------+--------+-----------+----------------------------------------------------------------+
|practice_003|user_003|session_003|d7489b48f8e588fd11608fbc2f72baafd5bdd62602321dd435f5c713adb17a82|
|practice_001|user_001|session_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|
|practice_002|user_002|session_002|da4bbec24d294be7e79f3037d748b8fb5c89cf0c7872a6a9b5f63b0169afb058|
+------------+--------+-----------+----------------------------------------------------------------+



In [31]:
pre_feedback_evidence_df = (
    pre_feedback_df.alias("f")
    .join(
        practice_taxonomy_df.alias("p"),
        (
            col("f.practice_id") == col("p.practice_id")
        )
        & (
            col("f.user_id") == col("p.user_id")
        ),
        "left"
    )

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("pre_feedback"),
                col("f.feedback_id"),
                col("p.taxonomy_id")
            ),
            256
        )
    )

    .withColumn("taxonomy_id", col("p.taxonomy_id"))
    .withColumn("event_id", lit(None).cast("string"))
    .withColumn("attempt_id", lit(None).cast("string"))
    .withColumn("insight_id", lit(None).cast("string"))
    .withColumn("validation_id", lit(None).cast("string"))

    .withColumn("evidence_type", lit("pre_feedback"))
    .withColumn("evidence_time", col("f.feedback_time"))

    .withColumn("is_correct", lit(None).cast("boolean"))
    .withColumn("score", lit(None).cast("float"))
    .withColumn("hints_used", lit(None).cast("int"))
    .withColumn("attempt_duration_seconds", lit(None).cast("int"))
    .withColumn("attempt_number", lit(None).cast("int"))

    .withColumn("extraction_confidence", lit(None).cast("float"))
    .withColumn("semantic_match_score", lit(None).cast("float"))
    .withColumn("reliability_score", lit(None).cast("float"))
    .withColumn("contradiction_flag", lit(None).cast("boolean"))

    .withColumn(
        "confidence_score",
        col("f.confidence_before_score")
    )
    .withColumn(
        "perceived_understanding_score",
        col("f.perceived_understanding_before_score")
    )
    .withColumn(
        "perceived_difficulty_score",
        col("f.expected_difficulty_score")
    )
    .withColumn(
        "still_confused",
        lit(None).cast("boolean")
    )

    .withColumn(
        "source_table",
        lit("silver_pre_practice_feedback")
    )
    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        col("evidence_id"),
        col("f.user_id").alias("user_id"),
        col("f.session_id").alias("session_id"),
        col("taxonomy_id"),

        col("event_id"),
        col("attempt_id"),
        col("f.feedback_id").alias("feedback_id"),
        col("insight_id"),
        col("validation_id"),

        col("evidence_type"),
        col("evidence_time"),

        col("is_correct"),
        col("score"),
        col("hints_used"),
        col("attempt_duration_seconds"),
        col("attempt_number"),

        col("extraction_confidence"),
        col("semantic_match_score"),
        col("reliability_score"),
        col("contradiction_flag"),

        col("confidence_score"),
        col("perceived_understanding_score"),
        col("perceived_difficulty_score"),
        col("still_confused"),

        col("source_table"),
        col("processing_time")
    )
)

In [32]:
pre_feedback_evidence_df.select(
    "evidence_id",
    "user_id",
    "feedback_id",
    "taxonomy_id",
    "evidence_type",
    "confidence_score",
    "perceived_understanding_score",
    "perceived_difficulty_score"
).show(truncate=False)

print("Pre-feedback evidence rows:", pre_feedback_evidence_df.count())

+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------------------+
|evidence_id                                                     |user_id |feedback_id |taxonomy_id                                                     |evidence_type|confidence_score|perceived_understanding_score|perceived_difficulty_score|
+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------------------+
|fa0deedd5ce73551266927424421de3934474b22a01ec0da79ad0dddd54278c3|user_001|feedback_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|pre_feedback |4               |3                            |8                         |
|efad45ffc5799f20b1caf69cddf6964

In [33]:
post_feedback_evidence_df = (
    post_feedback_df.alias("f")
    .join(
        practice_taxonomy_df.alias("p"),
        (
            col("f.practice_id") == col("p.practice_id")
        )
        & (
            col("f.user_id") == col("p.user_id")
        ),
        "left"
    )

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("post_feedback"),
                col("f.feedback_id"),
                col("p.taxonomy_id")
            ),
            256
        )
    )

    .withColumn("taxonomy_id", col("p.taxonomy_id"))
    .withColumn("event_id", lit(None).cast("string"))
    .withColumn("attempt_id", lit(None).cast("string"))
    .withColumn("insight_id", lit(None).cast("string"))
    .withColumn("validation_id", lit(None).cast("string"))

    .withColumn("evidence_type", lit("post_feedback"))
    .withColumn("evidence_time", col("f.feedback_time"))

    .withColumn("is_correct", lit(None).cast("boolean"))
    .withColumn("score", lit(None).cast("float"))
    .withColumn("hints_used", lit(None).cast("int"))
    .withColumn("attempt_duration_seconds", lit(None).cast("int"))
    .withColumn("attempt_number", lit(None).cast("int"))

    .withColumn("extraction_confidence", lit(None).cast("float"))
    .withColumn("semantic_match_score", lit(None).cast("float"))
    .withColumn("reliability_score", lit(None).cast("float"))
    .withColumn("contradiction_flag", lit(None).cast("boolean"))

    .withColumn(
        "confidence_score",
        col("f.confidence_after_score")
    )
    .withColumn(
        "perceived_understanding_score",
        col("f.perceived_understanding_after_score")
    )
    .withColumn(
        "perceived_difficulty_score",
        col("f.perceived_difficulty_score")
    )
    .withColumn(
        "still_confused",
        col("f.still_confused")
    )

    .withColumn(
        "source_table",
        lit("silver_post_practice_feedback")
    )
    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        col("evidence_id"),
        col("f.user_id").alias("user_id"),
        col("f.session_id").alias("session_id"),
        col("taxonomy_id"),

        col("event_id"),
        col("attempt_id"),
        col("f.feedback_id").alias("feedback_id"),
        col("insight_id"),
        col("validation_id"),

        col("evidence_type"),
        col("evidence_time"),

        col("is_correct"),
        col("score"),
        col("hints_used"),
        col("attempt_duration_seconds"),
        col("attempt_number"),

        col("extraction_confidence"),
        col("semantic_match_score"),
        col("reliability_score"),
        col("contradiction_flag"),

        col("confidence_score"),
        col("perceived_understanding_score"),
        col("perceived_difficulty_score"),
        col("still_confused"),

        col("source_table"),
        col("processing_time")
    )
)

In [34]:
post_feedback_evidence_df.select(
    "evidence_id",
    "user_id",
    "feedback_id",
    "taxonomy_id",
    "evidence_type",
    "confidence_score",
    "perceived_understanding_score",
    "perceived_difficulty_score",
    "still_confused"
).show(truncate=False)

print(
    "Post-feedback evidence rows:",
    post_feedback_evidence_df.count()
)

+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------------------+--------------+
|evidence_id                                                     |user_id |feedback_id |taxonomy_id                                                     |evidence_type|confidence_score|perceived_understanding_score|perceived_difficulty_score|still_confused|
+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------------------+--------------+
|cc7e638b07dc443cbb79142cbf76b1a625ca7841917aebe86ea7e8930a5b31de|user_001|feedback_002|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|post_feedback|5               |5                            |7              

In [35]:
check_in_topics_df = spark.table(
    "demo.silver.learner_check_in_topics"
)

check_in_topics_df.select(
    "feedback_id",
    "user_id",
    "session_id",
    "topic_id",
    "feedback_time",
    "perceived_understanding_score",
    "topic_confidence_score",
    "still_confused"
).show(truncate=False)

+------------+--------+-----------+--------------------+-------------------+-----------------------------+----------------------+--------------+
|feedback_id |user_id |session_id |topic_id            |feedback_time      |perceived_understanding_score|topic_confidence_score|still_confused|
+------------+--------+-----------+--------------------+-------------------+-----------------------------+----------------------+--------------+
|feedback_007|user_001|session_001|topic_virtual_memory|2026-07-23 18:00:00|4                            |4                     |true          |
|feedback_007|user_001|session_001|topic_recursion     |2026-07-23 18:00:00|7                            |7                     |false         |
|feedback_008|user_002|session_002|topic_recursion     |2026-07-24 17:30:00|5                            |5                     |true          |
|feedback_009|user_003|session_003|topic_memory_layout |2026-07-25 16:00:00|9                            |9                     |f

In [36]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    regexp_replace
)

check_in_topics_prepared_df = (
    check_in_topics_df
    .withColumn(
        "normalized_check_in_topic",
        lower(
            trim(
                regexp_replace(
                    regexp_replace(
                        col("topic_id"),
                        "^topic_",
                        ""
                    ),
                    "_",
                    " "
                )
            )
        )
    )
)

In [37]:
check_in_taxonomy_matches_df = (
    check_in_topics_prepared_df.alias("c")
    .join(
        taxonomy_matchable_df.alias("t"),
        col("c.normalized_check_in_topic")
        == col("t.normalized_match_name"),
        "left"
    )
    .select(
        col("c.feedback_id"),
        col("c.user_id"),
        col("c.session_id"),
        col("c.topic_id"),
        col("c.feedback_time"),
        col("c.perceived_understanding_score"),
        col("c.topic_confidence_score"),
        col("c.still_confused"),

        col("t.taxonomy_id"),
        col("t.taxonomy_level"),
        col("t.validation_status").alias(
            "taxonomy_validation_status"
        )
    )
)

In [38]:
check_in_taxonomy_matches_df.select(
    "feedback_id",
    "user_id",
    "topic_id",
    "taxonomy_id",
    "taxonomy_level",
    "taxonomy_validation_status",
    "perceived_understanding_score",
    "topic_confidence_score",
    "still_confused"
).orderBy(
    "feedback_id",
    "topic_id"
).show(truncate=False)

+------------+--------+--------------------+----------------------------------------------------------------+--------------+--------------------------+-----------------------------+----------------------+--------------+
|feedback_id |user_id |topic_id            |taxonomy_id                                                     |taxonomy_level|taxonomy_validation_status|perceived_understanding_score|topic_confidence_score|still_confused|
+------------+--------+--------------------+----------------------------------------------------------------+--------------+--------------------------+-----------------------------+----------------------+--------------+
|feedback_007|user_001|topic_recursion     |da4bbec24d294be7e79f3037d748b8fb5c89cf0c7872a6a9b5f63b0169afb058|subtopic      |approved                  |7                            |7                     |false         |
|feedback_007|user_001|topic_virtual_memory|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|subtopic   

In [39]:
check_in_evidence_df = (
    check_in_taxonomy_matches_df

    .withColumn(
        "evidence_id",
        sha2(
            concat_ws(
                "||",
                lit("check_in"),
                col("feedback_id"),
                col("taxonomy_id")
            ),
            256
        )
    )

    .withColumn(
        "event_id",
        lit(None).cast("string")
    )

    .withColumn(
        "attempt_id",
        lit(None).cast("string")
    )

    .withColumn(
        "insight_id",
        lit(None).cast("string")
    )

    .withColumn(
        "validation_id",
        lit(None).cast("string")
    )

    .withColumn(
        "evidence_type",
        lit("check_in")
    )

    .withColumn(
        "evidence_time",
        col("feedback_time")
    )

    .withColumn(
        "is_correct",
        lit(None).cast("boolean")
    )

    .withColumn(
        "score",
        lit(None).cast("float")
    )

    .withColumn(
        "hints_used",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_duration_seconds",
        lit(None).cast("int")
    )

    .withColumn(
        "attempt_number",
        lit(None).cast("int")
    )

    .withColumn(
        "extraction_confidence",
        lit(None).cast("float")
    )

    .withColumn(
        "semantic_match_score",
        lit(None).cast("float")
    )

    .withColumn(
        "reliability_score",
        lit(None).cast("float")
    )

    .withColumn(
        "contradiction_flag",
        lit(None).cast("boolean")
    )

    .withColumn(
        "confidence_score",
        col("topic_confidence_score")
    )

    .withColumn(
        "perceived_difficulty_score",
        lit(None).cast("int")
    )

    .withColumn(
        "source_table",
        lit("silver_learner_check_in_topics")
    )

    .withColumn(
        "processing_time",
        current_timestamp()
    )

    .select(
        "evidence_id",
        "user_id",
        "session_id",
        "taxonomy_id",

        "event_id",
        "attempt_id",
        "feedback_id",
        "insight_id",
        "validation_id",

        "evidence_type",
        "evidence_time",

        "is_correct",
        "score",
        "hints_used",
        "attempt_duration_seconds",
        "attempt_number",

        "extraction_confidence",
        "semantic_match_score",
        "reliability_score",
        "contradiction_flag",

        "confidence_score",
        "perceived_understanding_score",
        "perceived_difficulty_score",
        "still_confused",

        "source_table",
        "processing_time"
    )
)

In [40]:
check_in_evidence_df.select(
    "evidence_id",
    "user_id",
    "feedback_id",
    "taxonomy_id",
    "evidence_type",
    "confidence_score",
    "perceived_understanding_score",
    "still_confused",
    "source_table"
).show(truncate=False)

print(
    "Check-in evidence rows:",
    check_in_evidence_df.count()
)

print(
    "Distinct evidence IDs:",
    check_in_evidence_df
    .select("evidence_id")
    .distinct()
    .count()
)

+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------+------------------------------+
|evidence_id                                                     |user_id |feedback_id |taxonomy_id                                                     |evidence_type|confidence_score|perceived_understanding_score|still_confused|source_table                  |
+----------------------------------------------------------------+--------+------------+----------------------------------------------------------------+-------------+----------------+-----------------------------+--------------+------------------------------+
|6fafe77fbe4066391ecbf0554633e5575194f60c3fadc93198669066a35763e9|user_001|feedback_007|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|check_in     |4               |4                            |tru

In [41]:
all_evidence_df = (
    practice_evidence_df
    .unionByName(ai_insight_evidence_df)
    .unionByName(validated_insight_evidence_df)
    .unionByName(pre_feedback_evidence_df)
    .unionByName(post_feedback_evidence_df)
    .unionByName(check_in_evidence_df)
)

In [42]:
all_evidence_df.groupBy(
    "evidence_type"
).count().orderBy(
    "evidence_type"
).show()

+-----------------+-----+
|    evidence_type|count|
+-----------------+-----+
|       ai_insight|    9|
|         check_in|    5|
|    post_feedback|    3|
| practice_attempt|    5|
|     pre_feedback|    3|
|validated_insight|    9|
+-----------------+-----+



In [43]:
print(
    "Total evidence rows:",
    all_evidence_df.count()
)

print(
    "Distinct evidence IDs:",
    all_evidence_df
    .select("evidence_id")
    .distinct()
    .count()
)

Total evidence rows: 34
Distinct evidence IDs: 34


In [44]:
all_evidence_df.filter(
    col("evidence_id").isNull()
    | col("user_id").isNull()
    | col("taxonomy_id").isNull()
    | col("evidence_type").isNull()
    | col("evidence_time").isNull()
).show(truncate=False)

26/07/28 12:39:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+-------+----------+-----------+--------+----------+-----------+----------+-------------+-------------+-------------+----------+-----+----------+------------------------+--------------+---------------------+--------------------+-----------------+------------------+----------------+-----------------------------+--------------------------+--------------+------------+---------------+
|evidence_id|user_id|session_id|taxonomy_id|event_id|attempt_id|feedback_id|insight_id|validation_id|evidence_type|evidence_time|is_correct|score|hints_used|attempt_duration_seconds|attempt_number|extraction_confidence|semantic_match_score|reliability_score|contradiction_flag|confidence_score|perceived_understanding_score|perceived_difficulty_score|still_confused|source_table|processing_time|
+-----------+-------+----------+-----------+--------+----------+-----------+----------+-------------+-------------+-------------+----------+-----+----------+------------------------+--------------+-----------

In [45]:
invalid_taxonomy_links_df = (
    all_evidence_df.alias("e")
    .join(
        taxonomy_df.select("taxonomy_id").alias("t"),
        col("e.taxonomy_id") == col("t.taxonomy_id"),
        "left_anti"
    )
)

invalid_taxonomy_links_df.select(
    "evidence_id",
    "user_id",
    "taxonomy_id",
    "evidence_type"
).show(truncate=False)

print(
    "Invalid taxonomy links:",
    invalid_taxonomy_links_df.count()
)

+-----------+-------+-----------+-------------+
|evidence_id|user_id|taxonomy_id|evidence_type|
+-----------+-------+-----------+-------------+
+-----------+-------+-----------+-------------+

Invalid taxonomy links: 0


In [46]:
spark.sql("""
DELETE FROM demo.silver.learner_concept_evidence
""")

DataFrame[]

In [47]:
all_evidence_df.writeTo(
    "demo.silver.learner_concept_evidence"
).append()

In [48]:
spark.sql("""
SELECT
    evidence_type,
    COUNT(*) AS row_count
FROM demo.silver.learner_concept_evidence
GROUP BY evidence_type
ORDER BY evidence_type
""").show()

+-----------------+---------+
|    evidence_type|row_count|
+-----------------+---------+
|       ai_insight|        9|
|         check_in|        5|
|    post_feedback|        3|
| practice_attempt|        5|
|     pre_feedback|        3|
|validated_insight|        9|
+-----------------+---------+

